# 15 — Exploration-Exploitation Strategy Comparison

**Goal:** Justify Thompson Sampling by comparing it against 4 alternative bandit policies
for recommendation strategy selection in MARS.

**5 Policies:**

| Policy | Description |
|--------|-------------|
| Thompson Sampling | Beta(alpha, beta) posteriors — current MARS default |
| UCB1 | Upper Confidence Bound: mean + sqrt(2*ln(t)/n_i) |
| epsilon-greedy | Random strategy 10% of the time, best average 90% |
| Fixed (KB) | Always knowledge-based (no exploration) |
| Round-Robin | KB -> CB -> CF -> KB -> ... (deterministic) |

**Protocol:** 5 seeds x 5 policies. Per-step reward/regret tracking. Cold-start and
warm-start analysis. Per-cluster breakdown.

In [ ]:
"""Cell 1 — Imports and setup."""

from __future__ import annotations

import json
import logging
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.utils import set_global_seed, load_config

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("bandit_comparison")
logger.setLevel(logging.INFO)

SEEDS = [42, 123, 456, 789, 2024]
BANDIT_POLICIES = ["thompson", "ucb1", "epsilon_greedy", "fixed_kb", "fixed_round_robin"]
POLICY_LABELS = {
    "thompson": "Thompson Sampling",
    "ucb1": "UCB1",
    "epsilon_greedy": "ε-greedy (ε=0.1)",
    "fixed_kb": "Fixed (KB)",
    "fixed_round_robin": "Round-Robin",
}
POLICY_COLORS = {
    "thompson": "#1f77b4",
    "ucb1": "#2ca02c",
    "epsilon_greedy": "#ff7f0e",
    "fixed_kb": "#d62728",
    "fixed_round_robin": "#9467bd",
}
N_STEPS = 50  # recommendations per simulated user
RESULTS_DIR = ROOT / "results" / "bandit_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = load_config(str(ROOT / "configs" / "config.yaml"))

print(f"Policies: {list(POLICY_LABELS.values())}")
print(f"Seeds: {SEEDS}")
print(f"Steps per user: {N_STEPS}")

In [ ]:
"""Cell 2 — Load data and train shared agents."""

from data.loader import EdNetLoader
from data.preprocessor import EdNetPreprocessor
from agents.diagnostic_agent import DiagnosticAgent
from agents.confidence_agent import ConfidenceAgent
from agents.kg_agent import KnowledgeGraphAgent
from agents.prediction_agent import PredictionAgent
from agents.recommendation_agent import RecommendationAgent
from agents.personalization_agent import PersonalizationAgent, extract_user_features
from agents.orchestrator import Orchestrator

splits_dir = ROOT / "data" / "splits"
if (splits_dir / "train.parquet").exists():
    train_df = pd.read_parquet(splits_dir / "train.parquet")
    val_df = pd.read_parquet(splits_dir / "val.parquet")
    test_df = pd.read_parquet(splits_dir / "test.parquet")
else:
    loader = EdNetLoader(data_dir=str(ROOT / "data" / "raw"))
    interactions = loader.load_interactions(
        sample_users=CONFIG.get("data", {}).get("sample_users", 50000),
    )
    preprocessor = EdNetPreprocessor()
    splits = preprocessor.run(interactions, chunked=True)
    train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

loader = EdNetLoader(data_dir=str(ROOT / "data" / "raw"))
questions_df = loader.questions
lectures_df = loader.lectures

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Test users: {test_df['user_id'].nunique()}")

In [ ]:
"""Cell 3 — Bandit simulation engine using the actual RecommendationAgent."""

NUM_TAGS = 293
STRATEGIES = ["knowledge_based", "content_based", "collaborative"]


def parse_tags(tags):
    if isinstance(tags, list):
        return [int(t) for t in tags if 0 <= int(t) < NUM_TAGS]
    if isinstance(tags, (int, np.integer)):
        return [int(tags)] if 0 <= int(tags) < NUM_TAGS else []
    if isinstance(tags, str):
        return [int(t) for t in tags.split(";") if t.strip().isdigit() and 0 <= int(t) < NUM_TAGS]
    return []


def compute_reward(recommended_tags, ground_truth_fail_tags):
    """
    Reward = fraction of recommended tags that match actual knowledge gaps.
    Simulates: did the recommendation address the user's real needs?
    """
    if not ground_truth_fail_tags:
        return 0.0
    hits = len(set(recommended_tags) & set(ground_truth_fail_tags))
    return hits / max(len(ground_truth_fail_tags), 1)


def simulate_user_bandit(
    user_interactions, rec_agent, diag_agent, conf_agent, kg_agent,
    n_steps=N_STEPS, window=10,
):
    """
    Simulate n_steps recommendation rounds for a single user.

    At each step:
    1. Use context (interactions so far) to get recommendations
    2. Compute reward based on next `window` interactions (ground truth gaps)
    3. Update the bandit posterior

    Returns per-step: strategy, reward, regret, strategy distribution.
    """
    user_interactions = user_interactions.sort_values("timestamp")
    uid = str(user_interactions.iloc[0]["user_id"])

    if len(user_interactions) < n_steps + window:
        return None

    steps = []
    for step in range(n_steps):
        # Context: first (step+10) interactions (growing window)
        ctx_end = min(step + 10, len(user_interactions) - window)
        if ctx_end <= 0:
            continue
        ctx = user_interactions.iloc[:ctx_end]

        # Ground truth: next `window` interactions' failed tags
        gt_start = ctx_end
        gt_end = min(gt_start + window, len(user_interactions))
        future = user_interactions.iloc[gt_start:gt_end]
        gt_fail_tags = set()
        for _, row in future.iterrows():
            if not row["correct"]:
                gt_fail_tags.update(parse_tags(row["tags"]))

        if not gt_fail_tags:
            continue

        # Get recommendation
        n_interactions = len(ctx)
        strategy = rec_agent.select_strategy(uid, n_interactions=n_interactions)

        # Extract recommended tags based on strategy
        try:
            if strategy == "knowledge_based":
                diag_result = diag_agent.run_assessment(uid, ctx)
                conf_result = conf_agent.classify_batch(uid, ctx)
                kg_result = kg_agent.update_user_profile(uid, diag_result, conf_result)
                recs = rec_agent.get_knowledge_based(uid, kg_profile=kg_result, n=10)
            elif strategy == "content_based":
                recs = rec_agent.get_content_based(uid, gap_tags=list(gt_fail_tags)[:5], n=10)
            elif strategy == "collaborative":
                recs = rec_agent.get_collaborative(uid, n=10)
            else:
                recs = []

            rec_tags = set()
            for r in recs:
                rec_tags.update(r.related_tags)
        except Exception:
            rec_tags = set()

        # Compute reward and oracle reward
        reward = compute_reward(rec_tags, gt_fail_tags)

        # Oracle: best possible strategy
        oracle_reward = reward
        for alt_strat in STRATEGIES:
            if alt_strat == strategy:
                continue
            try:
                if alt_strat == "knowledge_based":
                    diag_result = diag_agent.run_assessment(uid, ctx)
                    conf_result = conf_agent.classify_batch(uid, ctx)
                    kg_result = kg_agent.update_user_profile(uid, diag_result, conf_result)
                    alt_recs = rec_agent.get_knowledge_based(uid, kg_profile=kg_result, n=10)
                elif alt_strat == "content_based":
                    alt_recs = rec_agent.get_content_based(uid, gap_tags=list(gt_fail_tags)[:5], n=10)
                elif alt_strat == "collaborative":
                    alt_recs = rec_agent.get_collaborative(uid, n=10)
                else:
                    alt_recs = []
                alt_tags = set()
                for r in alt_recs:
                    alt_tags.update(r.related_tags)
                alt_reward = compute_reward(alt_tags, gt_fail_tags)
                oracle_reward = max(oracle_reward, alt_reward)
            except Exception:
                pass

        regret = oracle_reward - reward

        # Update bandit
        rec_agent.update_reward(uid, strategy, reward)

        steps.append({
            "step": step,
            "strategy": strategy,
            "reward": reward,
            "regret": regret,
            "oracle_reward": oracle_reward,
            "n_interactions": n_interactions,
        })

    return steps


print("Simulation engine ready.")

In [ ]:
"""Cell 4 — Run simulation: 5 policies × 5 seeds × N users."""

from scripts.run_multi_seed import train_all_agents

all_bandit_results = {}  # policy -> [seed_result, ...]
cache_path = RESULTS_DIR / "bandit_raw.json"

# Select test users with enough interactions
test_user_groups = []
for uid, grp in test_df.groupby("user_id"):
    if len(grp) >= N_STEPS + 20:
        test_user_groups.append((uid, grp))
test_user_groups = test_user_groups[:200]  # Cap at 200 users for speed

print(f"Eligible test users: {len(test_user_groups)}")

for seed_idx, seed in enumerate(SEEDS):
    logger.info("=" * 60)
    logger.info("SEED %d/%d: %d", seed_idx + 1, len(SEEDS), seed)

    set_global_seed(seed)

    # Train shared agents once per seed
    agents, _ = train_all_agents(train_df, val_df, seed, CONFIG)
    diag = agents["diagnostic"]
    conf = agents["confidence"]
    kg = agents["knowledge_graph"]

    for policy in BANDIT_POLICIES:
        key = f"{policy}_seed{seed}"
        if key in all_bandit_results:
            continue

        logger.info("  Policy: %s", POLICY_LABELS[policy])
        t0 = time.time()

        # Create fresh RecommendationAgent with the desired bandit policy
        rec = RecommendationAgent(random_seed=seed, bandit_strategy=policy)
        if hasattr(rec, "initialize"):
            try:
                rec.initialize(
                    questions_df=questions_df,
                    lectures_df=lectures_df,
                    interactions_df=train_df,
                    train_user_ids=train_df["user_id"].unique().tolist(),
                )
            except Exception as e:
                logger.warning("  RecAgent init failed: %s", e)

        # Simulate all users
        all_steps = []
        for uid, grp in test_user_groups:
            user_steps = simulate_user_bandit(
                grp, rec, diag, conf, kg, n_steps=N_STEPS,
            )
            if user_steps:
                for s in user_steps:
                    s["user_id"] = str(uid)
                all_steps.extend(user_steps)

        elapsed = time.time() - t0

        # Aggregate
        if all_steps:
            steps_df = pd.DataFrame(all_steps)
            cum_reward = steps_df.groupby("step")["reward"].mean().cumsum().values
            cum_regret = steps_df.groupby("step")["regret"].mean().cumsum().values
            strategy_dist = steps_df.groupby("step")["strategy"].value_counts(normalize=True).unstack(fill_value=0)

            result = {
                "policy": policy,
                "seed": seed,
                "n_users": len(test_user_groups),
                "n_steps_total": len(all_steps),
                "cumulative_reward": cum_reward.tolist(),
                "cumulative_regret": cum_regret.tolist(),
                "avg_reward": float(steps_df["reward"].mean()),
                "avg_regret": float(steps_df["regret"].mean()),
                "total_reward": float(steps_df["reward"].sum()),
                "total_regret": float(steps_df["regret"].sum()),
                "strategy_distribution": strategy_dist.to_dict(),
                "time_s": round(elapsed, 1),
                # Cold-start (first 10 steps) and warm-start (steps 30+) metrics
                "cold_start_reward": float(steps_df[steps_df["step"] < 10]["reward"].mean()),
                "warm_start_reward": float(steps_df[steps_df["step"] >= 30]["reward"].mean()),
                "cold_start_regret": float(steps_df[steps_df["step"] < 10]["regret"].mean()),
                "warm_start_regret": float(steps_df[steps_df["step"] >= 30]["regret"].mean()),
            }
        else:
            result = {"policy": policy, "seed": seed, "n_steps_total": 0}

        all_bandit_results[key] = result
        logger.info("    reward=%.4f, regret=%.4f (%.1fs)",
                     result.get("avg_reward", 0), result.get("avg_regret", 0), elapsed)

# Save
with open(cache_path, "w") as f:
    json.dump(all_bandit_results, f, indent=2, default=str)

print(f"\nSimulation complete. {len(all_bandit_results)} runs saved.")

In [ ]:
"""Cell 5 — Summary table."""

# Group results by policy
policy_results = defaultdict(list)
for key, result in all_bandit_results.items():
    policy_results[result["policy"]].append(result)

summary_rows = []
for policy in BANDIT_POLICIES:
    results = policy_results[policy]
    if not results:
        continue
    row = {
        "Policy": POLICY_LABELS[policy],
        "Avg Reward": f"{np.mean([r['avg_reward'] for r in results]):.4f} ± "
                      f"{np.std([r['avg_reward'] for r in results], ddof=1):.4f}",
        "Avg Regret": f"{np.mean([r['avg_regret'] for r in results]):.4f} ± "
                      f"{np.std([r['avg_regret'] for r in results], ddof=1):.4f}",
        "Cold Reward": f"{np.mean([r.get('cold_start_reward', 0) for r in results]):.4f}",
        "Warm Reward": f"{np.mean([r.get('warm_start_reward', 0) for r in results]):.4f}",
        "Cold Regret": f"{np.mean([r.get('cold_start_regret', 0) for r in results]):.4f}",
        "Warm Regret": f"{np.mean([r.get('warm_start_regret', 0) for r in results]):.4f}",
    }

    # Wilcoxon vs Thompson Sampling
    if policy != "thompson":
        ts_rewards = [r["avg_reward"] for r in policy_results["thompson"]]
        pol_rewards = [r["avg_reward"] for r in results]
        if len(ts_rewards) == len(pol_rewards) >= 5:
            try:
                _, p = wilcoxon(ts_rewards, pol_rewards)
                row["p-value vs TS"] = f"{p:.4f}"
            except ValueError:
                row["p-value vs TS"] = "n/a"
        else:
            row["p-value vs TS"] = "n/a"
    else:
        row["p-value vs TS"] = "—"

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
print("=" * 100)
print("Exploration-Exploitation Strategy Comparison (mean ± std, 5 seeds)")
print("=" * 100)
display(summary)
summary.to_csv(RESULTS_DIR / "bandit_summary.csv", index=False)
summary.to_latex(RESULTS_DIR / "bandit_summary.tex", index=False, escape=False)

In [ ]:
"""Cell 6 — Visualization A: Regret curves (cumulative regret vs step)."""

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# --- Panel A: Cumulative Regret ---
ax = axes[0]
for policy in BANDIT_POLICIES:
    results = policy_results[policy]
    if not results:
        continue
    # Align curves to same length
    min_len = min(len(r["cumulative_regret"]) for r in results if "cumulative_regret" in r)
    if min_len == 0:
        continue
    curves = np.array([r["cumulative_regret"][:min_len] for r in results if "cumulative_regret" in r])
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)
    steps = np.arange(min_len)

    ax.plot(steps, mean_curve, label=POLICY_LABELS[policy],
            color=POLICY_COLORS[policy], linewidth=2)
    ax.fill_between(steps, mean_curve - std_curve, mean_curve + std_curve,
                    color=POLICY_COLORS[policy], alpha=0.15)

ax.set_xlabel("Step", fontsize=11)
ax.set_ylabel("Cumulative Regret", fontsize=11)
ax.set_title("(a) Cumulative Regret vs Step", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Panel B: Cumulative Reward ---
ax2 = axes[1]
for policy in BANDIT_POLICIES:
    results = policy_results[policy]
    if not results:
        continue
    min_len = min(len(r["cumulative_reward"]) for r in results if "cumulative_reward" in r)
    if min_len == 0:
        continue
    curves = np.array([r["cumulative_reward"][:min_len] for r in results if "cumulative_reward" in r])
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)
    steps = np.arange(min_len)

    ax2.plot(steps, mean_curve, label=POLICY_LABELS[policy],
             color=POLICY_COLORS[policy], linewidth=2)
    ax2.fill_between(steps, mean_curve - std_curve, mean_curve + std_curve,
                     color=POLICY_COLORS[policy], alpha=0.15)

ax2.set_xlabel("Step", fontsize=11)
ax2.set_ylabel("Cumulative Reward", fontsize=11)
ax2.set_title("(b) Cumulative Reward vs Step", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# --- Panel C: Cold-start vs Warm-start comparison ---
ax3 = axes[2]
x = np.arange(len(BANDIT_POLICIES))
width = 0.35

cold_means = []
warm_means = []
labels = []
for policy in BANDIT_POLICIES:
    results = policy_results[policy]
    if not results:
        cold_means.append(0)
        warm_means.append(0)
    else:
        cold_means.append(np.mean([r.get("cold_start_reward", 0) for r in results]))
        warm_means.append(np.mean([r.get("warm_start_reward", 0) for r in results]))
    labels.append(POLICY_LABELS[policy])

bars1 = ax3.bar(x - width/2, cold_means, width, label="Cold-start (steps 0-9)",
                color="#ff9999", edgecolor="black", linewidth=0.5)
bars2 = ax3.bar(x + width/2, warm_means, width, label="Warm-start (steps 30+)",
                color="#66b3ff", edgecolor="black", linewidth=0.5)

ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax3.set_ylabel("Average Reward", fontsize=11)
ax3.set_title("(c) Cold-start vs Warm-start", fontsize=12, fontweight="bold")
ax3.legend(fontsize=9)
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "regret_curves.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "regret_curves.pdf", bbox_inches="tight")
plt.show()
print("Saved: regret_curves.png / .pdf")

In [ ]:
"""Cell 7 — Visualization B: Strategy distribution over time (stacked area)."""

fig, axes = plt.subplots(1, len(BANDIT_POLICIES), figsize=(20, 4), sharey=True)

for ax, policy in zip(axes, BANDIT_POLICIES):
    results = policy_results[policy]
    if not results or "strategy_distribution" not in results[0]:
        ax.set_title(POLICY_LABELS[policy])
        continue

    # Average strategy distribution across seeds
    all_dists = []
    for r in results:
        dist = r["strategy_distribution"]
        df_dist = pd.DataFrame(dist)
        all_dists.append(df_dist)

    if not all_dists:
        ax.set_title(POLICY_LABELS[policy])
        continue

    # Align and average
    min_steps = min(len(d) for d in all_dists)
    aligned = [d.iloc[:min_steps].values for d in all_dists]
    avg_dist = np.mean(aligned, axis=0)
    cols = all_dists[0].columns.tolist()

    strat_colors = {
        "knowledge_based": "#1f77b4",
        "content_based": "#ff7f0e",
        "collaborative": "#2ca02c",
    }

    ax.stackplot(
        range(min_steps),
        *[avg_dist[:, i] for i in range(avg_dist.shape[1])],
        labels=cols,
        colors=[strat_colors.get(c, "gray") for c in cols],
        alpha=0.8,
    )
    ax.set_title(POLICY_LABELS[policy], fontsize=10, fontweight="bold")
    ax.set_xlabel("Step")
    if ax == axes[0]:
        ax.set_ylabel("Strategy Proportion")
    ax.set_ylim(0, 1)

axes[-1].legend(loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=8)

plt.suptitle("Strategy Distribution Over Time by Policy", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "strategy_distribution.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "strategy_distribution.pdf", bbox_inches="tight")
plt.show()
print("Saved: strategy_distribution.png / .pdf")

## Key Findings (expected)

1. **Thompson Sampling achieves lowest regret** — principled exploration balances KB/CB/CF
2. **UCB1 close second** — but slower convergence due to deterministic exploration bonus
3. **e-greedy underexplores** — 10% random is crude vs TS's adaptive exploration
4. **Fixed KB good at cold-start** — but plateaus as it never discovers CB/CF opportunities
5. **Round-Robin wastes budget** — forces equal allocation regardless of user profile
6. **TS converges faster** — stabilizes on best strategy within ~15-20 steps
7. **Cold-start**: TS explores more aggressively early, adapts faster to new users

In [ ]:
"""Cell 8 — Save all outputs."""

final_meta = {
    "experiment": "MARS Exploration-Exploitation Strategy Comparison",
    "seeds": SEEDS,
    "policies": list(POLICY_LABELS.values()),
    "n_steps_per_user": N_STEPS,
    "n_users": len(test_user_groups),
    "output_files": [
        str(RESULTS_DIR / "bandit_raw.json"),
        str(RESULTS_DIR / "bandit_summary.csv"),
        str(RESULTS_DIR / "bandit_summary.tex"),
        str(RESULTS_DIR / "regret_curves.png"),
        str(RESULTS_DIR / "strategy_distribution.png"),
    ],
}

with open(RESULTS_DIR / "bandit_meta.json", "w") as f:
    json.dump(final_meta, f, indent=2)

print("Bandit comparison complete.")
print(f"Results: {RESULTS_DIR}")